In [1]:
# =========================================================================
# CÉLULA 1: CARREGAMENTO DE DADOS (AJUSTADO PARA A SUA PASTA)
# =========================================================================
import pandas as pd
import os

print("--- FASE 1: CARREGAMENTO ---")

# Como você está em C:\Users\User e a pasta Data está lá também:
BASE_PATH = "Data" 

# Construindo os caminhos
FILE_PATH_1 = os.path.join(BASE_PATH, "Base Artigo 1.csv")
FILE_PATH_2 = os.path.join(BASE_PATH, "Base Artigo 2.csv")
FILE_PATH_3 = os.path.join(BASE_PATH, "Base Artigo 3.csv")

# Carregar os três arquivos CSV
try:
    df1 = pd.read_csv(FILE_PATH_1, sep=',')
    df2 = pd.read_csv(FILE_PATH_2, sep=',')
    df3 = pd.read_csv(FILE_PATH_3, sep=',')
    print("Arquivos carregados com sucesso!")
    load_successful = True
except FileNotFoundError as e:
    print(f"ERRO: Não encontrei o ficheiro. Verifique se ele está dentro da pasta Data.")
    print(f"Detalhe do erro: {e}")

--- FASE 1: CARREGAMENTO ---
Arquivos carregados com sucesso!


In [6]:
# ==============================================================================
# CÉLULA 2: PRÉ-PROCESSAMENTO (CORRIGIDA COM IMPORTS E SEM AVISOS)
# ==============================================================================
import pandas as pd
import numpy as np  # <--- Isto resolve o erro do 'np'
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("\n--- FASE 2: PRÉ-PROCESSAMENTO E SELEÇÃO DE FEATURES ---")

# 1. UNIÃO E FILTRAGEM
df_combined = pd.concat([df1, df2, df3], ignore_index=True)

if 'Numero' in df_combined.columns:
    df_combined.drop(columns=['Numero'], inplace=True)

TARGET_COL = 'Entorse'

COLUNAS_SELECIONADAS = [
    TARGET_COL,
    'T0_T1_Match_Time_exposure',
    'T0_T1_Training_Time_exposure',
    'T0SRTMax',
    'T0TTestMin',
    'T0SJmMax',
    'T0Veli',
    'T0RazaoAADto',
    'T0RazaoAAEsq',
]

# Filtra colunas que realmente existem
cols_to_keep = list(set(COLUNAS_SELECIONADAS) & set(df_combined.columns))
df_filtered = df_combined[cols_to_keep].copy()

# 2. TRATAMENTO DE VALORES OMISSOS (Versão moderna para evitar o ChainedAssignmentError)
# Em vez de inplace=True, atribuímos diretamente à coluna
df_filtered[TARGET_COL] = df_filtered[TARGET_COL].fillna(0).astype(int)

# Identifica colunas numéricas (Agora o 'np' já funciona!)
numerical_cols = df_filtered.select_dtypes(include=[np.number]).columns.tolist()
if TARGET_COL in numerical_cols:
    numerical_cols.remove(TARGET_COL)

# Imputação de NaN (Mediana para numéricas)
for col in numerical_cols:
    df_filtered[col] = df_filtered[col].fillna(df_filtered[col].median())

# 3. DEFINIÇÃO DE X/Y E CODIFICAÇÃO
Y = df_filtered[TARGET_COL] 
X = df_filtered.drop(columns=[TARGET_COL]) 
X = pd.get_dummies(X, drop_first=True) 

print(f"Dimensão de X após codificação: {X.shape}")

# 4. DIVISÃO TREINO/TESTE E NORMALIZAÇÃO
# Agora o train_test_split também vai funcionar porque importámos acima
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.75, random_state=42, stratify=Y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("--- ✅ FASE 2 CONCLUÍDA! DADOS PRONTOS. ---")


--- FASE 2: PRÉ-PROCESSAMENTO E SELEÇÃO DE FEATURES ---
Dimensão de X após codificação: (181, 56)
--- ✅ FASE 2 CONCLUÍDA! DADOS PRONTOS. ---


In [7]:
# ==============================================================================
# CÉLULA 3: TREINO DO MODELO COM RANDOM FOREST
# ==============================================================================
from sklearn.ensemble import RandomForestClassifier

print("\n--- FASE 3: CONSTRUÇÃO E TREINO DO MODELO RANDOM FOREST ---")

# 1. CONSTRUÇÃO DO MODELO RANDOM FOREST
model_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    max_depth=5
)

# 2. TREINO DO MODELO
model_rf.fit(X_train_scaled.values, y_train)

print("\n--- ✅ FASE 3 CONCLUÍDA! MODELO TREINADO. ---")


--- FASE 3: CONSTRUÇÃO E TREINO DO MODELO RANDOM FOREST ---

--- ✅ FASE 3 CONCLUÍDA! MODELO TREINADO. ---


In [8]:
# ==============================================================================
# CÉLULA 4: AVALIAÇÃO FINAL DO DESEMPENHO (RANDOM FOREST)
# ==============================================================================
from sklearn.metrics import recall_score, f1_score, accuracy_score, confusion_matrix
import time

print("\n--- FASE 4: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (RANDOM FOREST) ---")

# ⏱️ Início da medição do tempo
start_time = time.perf_counter()

# 1. PREVISÃO NOS DADOS DE TESTE (SÓ FEATURES!)
y_pred = model_rf.predict(X_test_scaled.values)

# 2. CÁLCULO DAS MÉTRICAS
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f_score = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

# ⏱️ Fim da medição do tempo
end_time = time.perf_counter()
execution_time_ms = (end_time - start_time) * 1000

print("\n=== RESULTADOS FINAIS DE DESEMPENHO ===")
print(f"Precisão (Accuracy): {accuracy:.4f}")
print(f"Recall (Sensibilidade): {recall:.4f}")
print(f"F-Score: {f_score:.4f}")
print(f"Tempo de Execução: {execution_time_ms:.2f} ms")

print("\nMatriz de Confusão:")
print(conf_matrix)

# 3. FALSOS NEGATIVOS (ERRO CRÍTICO)
if conf_matrix.shape == (2, 2):
    fn = conf_matrix[1, 0]
    # Este FN é o número que o professor quer ver a ZERO
    print(f"\nFalsos Negativos (FN - Entorse real não prevista): {fn}")

print("\n--- ✅ PROJETO CONCLUÍDO E VALIDADO (RANDOM FOREST) ---")



--- FASE 4: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (RANDOM FOREST) ---

=== RESULTADOS FINAIS DE DESEMPENHO ===
Precisão (Accuracy): 0.5882
Recall (Sensibilidade): 0.6393
F-Score: 0.5821
Tempo de Execução: 24.47 ms

Matriz de Confusão:
[[39 22]
 [34 41]]

Falsos Negativos (FN - Entorse real não prevista): 34

--- ✅ PROJETO CONCLUÍDO E VALIDADO (RANDOM FOREST) ---
